# Gold Layer — Model Training & State Classification

Welcome to the **Gold Layer** — the final stage of the pipeline. Everything
before this point prepared the data; now we train the classifier that makes
the actual decision.

## Where Are We in the Pipeline?

```
Raw Screenshot  ──►  [BRONZE]  ──►  [SILVER]  ──►  [GOLD]  ──►  Prediction
                      (denoise,      (extract      (you are here)
                       sharpen)       features)     (train model)
```

- **Bronze** cleaned the pixels.
- **Silver** extracted structured game features (player health, enemy health,
  attack/defense state).
- **Gold** takes those features and answers the question: is the player
  **winning**, **losing**, or in a **stalemate**?


## Setup: Imports & Paths

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.manifold import TSNE

from src.config import BRONZE_DIR, SILVER_DIR
from src.pipeline.bronze import process_image as bronze_process
from src.pipeline.gold import (
    FEATURE_NAMES,
    STATE_LABELS,
    features_to_vector,
    generate_synthetic_dataset,
    load_labeled_dataset,
    predict,
    rule_based_label,
    train_and_compare,
    train_gold_model,
)
from src.pipeline.silver import process_image as silver_process

sns.set_theme(style="darkgrid")
%matplotlib inline

---

# Part 1: The Classification Problem

The Gold layer ignores pixels entirely. It consumes the structured feature
vector produced by the Silver layer:

| # | Feature             | Meaning                                        |
|---|---------------------|------------------------------------------------|
| 1 | `player_health`     | Player's remaining health (0..1)               |
| 2 | `mean_enemy_health` | Average health of detected enemies             |
| 3 | `min_enemy_health`  | Health of the weakest enemy                    |
| 4 | `health_ratio`      | player_health / mean_enemy_health              |
| 5 | `num_enemies`       | How many enemies are on screen                 |
| 6 | `attacking`         | Is the player attacking? (0/1)                 |
| 7 | `defending`         | Is the player defending? (0/1)                 |
| 8 | `damage_indicator`  | Is the player taking damage? (0/1)             |

Every screenshot becomes one 8-dimensional vector. The classifier must map
that vector to one of three plain-English labels:

- **winning** — the player has the advantage (health ratio + aggression)
- **losing** — the player is at a disadvantage (low health, taking damage)
- **stalemate** — roughly even

### Why a separate classifier?

The Silver CNN answers *"what is on screen?"* (low-level perception). The Gold
classifier answers *"who is winning?"* (high-level reasoning). Two different
problems, two separate models — so each can be debugged independently.


---

# Part 2: The Rule-Based Labeler (Free Labels)

Before any model exists, we need **labeled** data. Until human labels arrive,
`rule_based_label()` generates them deterministically from the features,
mirroring the AGENTS.md definition of each state.

The rules are checked in priority order:

1. **losing** — player health < 0.2, OR health ratio <= 0.7 (enemy much
   healthier), OR the player is taking damage below half health
2. **winning** — health ratio >= 1.3 while not defending, OR attacking with
   a positive health ratio (> 1.0)
3. **stalemate** — everything else

These rules encode the domain knowledge from the design document — and they
are also a great way to *bootstrap* a training set: run the rules over
unlabeled Silver data, then have a human verify the results instead of
labeling from scratch.

In [ ]:
winning_example = {
    "image_path": "/synthetic/frame_0000.png",
    "player_health": 0.8,
    "player_position": [0.5, 0.5],
    "enemies": [{"bbox": [100, 200, 60, 60], "health": 0.3,
                 "health_bar_bbox": [100, 170, 50, 8], "confidence": 0.9}],
    "attacking": True,
    "defending": False,
    "damage_indicator": False,
    "num_enemies": 1,
}

losing_example = {
    "image_path": "/synthetic/frame_0001.png",
    "player_health": 0.2,
    "player_position": [0.5, 0.5],
    "enemies": [{"bbox": [100, 200, 60, 60], "health": 0.9,
                 "health_bar_bbox": [100, 170, 50, 8], "confidence": 0.9}],
    "attacking": False,
    "defending": True,
    "damage_indicator": True,
    "num_enemies": 1,
}

stalemate_example = {
    "image_path": "/synthetic/frame_0002.png",
    "player_health": 0.5,
    "player_position": [0.5, 0.5],
    "enemies": [{"bbox": [100, 200, 60, 60], "health": 0.5,
                 "health_bar_bbox": [100, 170, 50, 8], "confidence": 0.9}],
    "attacking": False,
    "defending": False,
    "damage_indicator": False,
    "num_enemies": 1,
}

for name, example in [("winning", winning_example),
                      ("losing", losing_example),
                      ("stalemate", stalemate_example)]:
    vec = features_to_vector(example)
    print(f"{name:10s} -> {rule_based_label(example):10s} "
          f"health_ratio={vec[FEATURE_NAMES.index('health_ratio')]:.3f}")

In [ ]:
vector = features_to_vector(winning_example)
print("Feature vector:", vector)
print()
for name, value in zip(FEATURE_NAMES, vector):
    print(f"  {name:18s} {value:.4f}")

---

# Part 3: Generating a Synthetic Labeled Dataset

`generate_synthetic_dataset()` creates a fully labeled dataset for pipeline
testing. Every sample is a SilverFeatures dict; the label comes from the
rule-based labeler, and the classes are **stratified** so all three states
always appear.

```
dataset/
    silver/*.json    # feature dicts + embedded "label" key
    labels.csv       # stem,label — the human-editable source of truth
```

This is exactly the layout your real labeled data will use — swap the
synthetic JSONs for real ones and nothing else changes.

In [ ]:
dataset_root = generate_synthetic_dataset(
    str(Path(tempfile.mkdtemp()) / "ds"), num_samples=300, seed=42)
print(f"Dataset written to: {dataset_root}")

X, y, stems = load_labeled_dataset(dataset_root)
labels = pd.Series([STATE_LABELS[i] for i in y])
print(f"\nFeature matrix: {X.shape}  (300 screenshots x {X.shape[1]} features)")
print("\nClass balance:")
print(labels.value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

labels.value_counts().plot(kind="bar", ax=axes[0],
                           color=["#2ca02c", "#d62728", "#1f77b4"])
axes[0].set_title("Class Balance (stratified by construction)",
                  fontsize=13, fontweight="bold")
axes[0].set_ylabel("Samples")
axes[0].set_xlabel("State")
axes[0].tick_params(axis="x", rotation=0)

feature_df = pd.DataFrame(X, columns=FEATURE_NAMES)
corr = feature_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            ax=axes[1], square=True,
            cbar_kws={"label": "Pearson correlation"})
axes[1].set_title("Feature Correlation Matrix",
                  fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

---

# Part 4: Real Data — How Labels Get Attached

For real screenshots you have two ways to attach a label to each Silver JSON:

1. **labels.csv** — one `stem,label` row per screenshot. Fill this in a
   spreadsheet while reviewing screenshots.
2. **Embedded `label` key** — add `"label": "winning"` directly inside the
   Silver JSON.

`load_labeled_dataset()` reads both: the CSV **overrides** the embedded label
(so you can re-label without touching the JSONs), and any sample without a
label raises a clear error naming the missing stem.

Let's inspect the labels.csv the synthetic generator produced:

In [ ]:
labels_csv = Path(dataset_root) / "labels.csv"
print(pd.read_csv(labels_csv).head(8).to_string(index=False))
print(f"\n{len(list(Path(dataset_root).glob('silver/*.json')))} silver JSONs, "
      f"{len(pd.read_csv(labels_csv))} label rows")

X, y, stems = load_labeled_dataset(dataset_root)
print(f"Loaded: X={X.shape}, y={y.shape}")

---

# Part 5: Training One Classifier with MLflow

Every training run is tracked with **MLflow**:

- **Parameters** — model name, seed, split sizes, class distribution
- **Metrics** — accuracy, macro-F1, weighted-F1
- **Artifacts** — confusion matrix PNG, classification report, saved model

`train_gold_model()` trains any single model from the zoo:

In [ ]:
result = train_gold_model(
    dataset_root,
    model_name="random_forest",
    experiment_name="gold_demo",
    run_name="demo_random_forest",
    val_split=0.2,
    seed=42,
)
print(f"\nRun ID    : {result['run_id']}")
print(f"Accuracy  : {result['accuracy']:.4f}")
print(f"Macro F1  : {result['macro_f1']:.4f}")
print(f"Model at  : {result['model_path']}")

## Predicting with the Trained Model

`predict()` takes either a SilverFeatures dict or a raw feature vector and
returns the predicted label plus probabilities for **all three** states.

In [ ]:
import mlflow

model = mlflow.sklearn.load_model(result["model_path"])

examples = [("winning", winning_example),
            ("losing", losing_example),
            ("stalemate", stalemate_example)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (true_label, example) in zip(axes, examples):
    predicted, probs = predict(example, model)
    bars = ax.bar(list(probs.keys()), list(probs.values()),
                  color=["#2ca02c", "#d62728", "#1f77b4"])
    ax.set_title(f"True: {true_label}  →  Predicted: {predicted}",
                 fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Probability")
    for bar, value in zip(bars, probs.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{value:.2f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

---

# Part 6: The Model Zoo — Finding the Best Fit

AGENTS.md requires MLflow to test a **wide variety of models**. The zoo:

| Model                 | Why it might win                    |
|-----------------------|-------------------------------------|
| `logistic_regression` | Baseline; the features are simple   |
| `random_forest`       | Handles threshold boundaries well   |
| `gradient_boosting`   | Strong on small tabular data        |
| `xgboost`             | State-of-the-art gradient boosting  |
| `svc`                 | Great with scaled, separable data   |
| `pytorch_mlp`         | Deep learning entry in the pipeline |

`train_and_compare()` trains all six, logs one MLflow run per model, then
writes a **leaderboard** run containing the comparison table, the feature
correlation matrix, and a t-SNE embedding of the feature space.

In [ ]:
leaderboard = train_and_compare(
    dataset_root,
    experiment_name="gold_demo",
    model_params=None,  # each model's default hyperparameters
    val_split=0.2,
    seed=42,
)
print(leaderboard[["model_name", "accuracy", "macro_f1", "weighted_f1"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=leaderboard, x="model_name", y="macro_f1", ax=ax,
            order=leaderboard["model_name"], palette="viridis")
ax.set_title("Macro-F1 by Model (sorted by performance)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Macro F1")
ax.set_xlabel("Model")
ax.set_ylim(0, 1.05)
for i, row in enumerate(leaderboard.itertuples()):
    ax.text(i, row.macro_f1 + 0.02, f"{row.macro_f1:.3f}",
            ha="center", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Best model: {leaderboard.iloc[0]['model_name']} "
      f"(macro_f1={leaderboard.iloc[0]['macro_f1']:.3f})")

In [ ]:
tsne = TSNE(n_components=2,
            perplexity=min(30, max(5, len(X) // 5)), random_state=42)
embedding = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 7))
colors = {"winning": "#2ca02c", "losing": "#d62728", "stalemate": "#1f77b4"}
for state in STATE_LABELS:
    mask = labels == state
    ax.scatter(embedding[mask, 0], embedding[mask, 1], s=50, alpha=0.8,
               label=state, color=colors[state])
ax.set_title("t-SNE Embedding of the Gold Feature Space",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---

# Part 7: The Complete Pipeline on a Real Screenshot

Now the full journey: a raw screenshot from `data/bronze/` goes through
**Bronze → Silver → Gold**, and the trained model makes the final call.

> Note: the Silver model isn't trained on real data yet, so the Silver layer
> returns default features. The Gold prediction is therefore based on those
> defaults — the mechanics are correct, but real accuracy needs real data.

In [ ]:
pngs = sorted(BRONZE_DIR.glob("*.png"))
if not pngs:
    raise FileNotFoundError(f"No PNGs found in {BRONZE_DIR}")

real_image = str(pngs[0])
print(f"Input : {pngs[0].name}")

bronze_result = bronze_process(real_image, str(SILVER_DIR))
silver_result = silver_process(real_image, str(SILVER_DIR))

features = silver_result["silver_features"]
print(f"Silver: player_health={features['player_health']:.2f}, "
      f"enemies={features['num_enemies']}, attacking={features['attacking']}")

# Two opinions: the deterministic rule and the trained model.
rule_label = rule_based_label(features)
predicted, probs = predict(features, model)

print(f"\nRule-based reference : {rule_label.upper()} "
      f"(health=0.0 is clearly losing)")
print(f"Model prediction    : {predicted.upper()} "
      f"(winning={probs['winning']:.2f})")
print("\nThe model disagrees because the default features (all zeros) are "
      "out of distribution — the synthetic training data never contained a "
      "screenshot with zero health. Low-confidence, OOD predictions are a "
      "signal to collect more real data, not to trust the model.")

---

# Part 8: Exploring the Results in the MLflow UI

Open the MLflow dashboard to compare runs and inspect artifacts:

```
mlflow ui
```

then visit http://localhost:5000.

For the `gold_demo` experiment you will find:

- Six model runs, each with a **confusion matrix** and a **classification
  report** — don't just log numbers, look at where each model gets confused
- A `leaderboard` run containing:
  - `leaderboard.csv` — the ranked comparison table
  - `feature_correlations.png` — how features relate to each other
  - `tsne_embedding.png` — the feature space colored by state

This is your **human-in-the-loop** loop: spot a failure mode in the confusion
matrix, collect more screenshots of that state, re-label, retrain, repeat.

---

## Summary

- The Gold layer classifies `winning` / `losing` / `stalemate` from 8 Silver
  features — no pixels involved.
- `rule_based_label()` provides deterministic bootstrap labels until human
  labels exist.
- `train_gold_model()` logs every run to MLflow (params, metrics, confusion
  matrix, classification report).
- `train_and_compare()` races all six models and returns a ranked leaderboard
  plus correlation / t-SNE visualizations.
- `predict()` turns any trained model into a live game-state predictor.

## Next Steps

The bottleneck is now **data**, not code: the project needs 1000–2000 labeled
real screenshots. See the data-collection plan in `docs/` for how to capture
and label them as efficiently as possible — then retrain with:

```python
train_and_compare("data/gold", experiment_name="gold_production")
```